<a href="https://colab.research.google.com/github/labphonlab/phonetics-research-statistics-support/blob/main/notebooks/ch13_power_sample_size.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

このバッジから開くと、設定なしでそのまま実行できます。
書き換えを残したいときは **ファイル → ドライブにコピーを保存** を一度行ってください。


# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第13回：検定力とサンプルサイズをシミュレーションで決める


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本ファイルの公開用URLを、みだりに他の人へ共有しないこと。

本ファイルの公開用URLにアクセスすると、お手元のGoogleアカウントに本ファイルのコピーが作成される。
そのコピーはあなただけのファイルとして生成されるため、自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照: Brysbaert & Stevens (2018) 混合効果モデルの検定力／Hoenig & Heisey (2001) 事後検定力の誤用／Westfall, Kenny & Judd (2014) 交差ランダム効果デザインの検定力。

In [ ]:
install.packages(c("lme4", "lmerTest"), repos = "https://cloud.r-project.org")
library(lme4)
library(lmerTest)
library(tidyverse)


■ 「何人録ればいいですか」に答える：

研究を始める前に必ず出てくる問いが、**話者を何人、語を何語用意すればよいか**である。この問いに勘で答えると、2つの失敗のどちらかが起きる。少なすぎれば、効果が実在しても検出できずに「差は無かった」と書くことになる。多すぎれば、実質的に意味のない差まで有意になり、録音と分析にかけた労力も無駄になる。

混合効果モデルの検定力は、教科書の公式では計算できない。話者のランダム切片・ランダム傾き・項目のランダム切片・残差という複数の分散が絡むためである。そこで実務では**シミュレーション**で求める。考え方は単純で、「想定する条件でデータを何百組も作り、そのうち何割で有意になるか」を数えるだけである。

■ 検定力を計算する関数を作る：

In [ ]:
# 引数で「想定する世界」を指定し、その世界で何割の実験が成功するかを返す。
# effect  : 検出したい効果の大きさ（ms）
# sd_sp   : 話者ごとの全体水準のばらつき
# sd_slope: 話者ごとの「効果の大きさ」自体のばらつき（ランダム傾き）
# sd_it   : 項目ごとのばらつき
# sd_res  : 残差
simulate_power <- function(n_speaker, n_item, effect,
                           sd_sp = 8, sd_slope = 6, sd_it = 4, sd_res = 7,
                           nsim = 200, seed = 1) {
  set.seed(seed)
  # 条件は「語の性質」として割り当てる（第1回のvoiced/voicelessと同じ設計）
  cond_of_item <- rep(c(-0.5, 0.5), length.out = n_item)

  out <- replicate(nsim, {
    sp_int   <- rnorm(n_speaker, 0, sd_sp)
    sp_slope <- rnorm(n_speaker, 0, sd_slope)
    it_int   <- rnorm(n_item,    0, sd_it)

    d <- expand.grid(sp = 1:n_speaker, it = 1:n_item)
    d$cond <- cond_of_item[d$it]
    d$y <- 50 + effect * d$cond +
           sp_int[d$sp] + sp_slope[d$sp] * d$cond +
           it_int[d$it] + rnorm(nrow(d), 0, sd_res)
    d$sp <- factor(d$sp); d$it <- factor(d$it)

    # ★収束の警告・singular fitを黙って揉み消さない。異常フィットを
    #   別カウントし、p値だけを見て成功/失敗をカウントすることの危険を避ける。
    anomaly <- FALSE
    m <- withCallingHandlers(
      tryCatch(lmer(y ~ cond + (1 + cond | sp) + (1 | it), data = d),
               error = function(e) NULL),
      warning = function(w) { anomaly <<- TRUE; invokeRestart("muffleWarning") }
    )
    if (is.null(m)) return(c(sig = NA, anomaly = 1))
    if (isSingular(m)) anomaly <- TRUE
    sig <- coef(summary(m))["cond", "Pr(>|t|)"] < 0.05
    c(sig = as.numeric(sig), anomaly = as.numeric(anomaly))
  })

  sig_vals <- out["sig", ]
  n_valid <- sum(!is.na(sig_vals))
  power <- mean(sig_vals, na.rm = TRUE)
  # モンテカルロ標準誤差。nsim=200程度では検定力0.5〜0.6付近でSE≈0.03〜0.04になる
  # ——3桁目まで表示するのは過剰な精度である。2桁までにとどめる。
  se <- sqrt(power * (1 - power) / n_valid)
  list(power = power, se = se, anomaly_rate = mean(out["anomaly", ]))
}

# ★健全性チェック: 各話者が両方の条件を持っているかを必ず確認する。
#   ここを間違えると（条件が話者と交絡すると）、検定力の見積もりが丸ごと狂う。
check <- expand.grid(sp = 1:4, it = 1:6)
check$cond <- rep(c(-0.5, 0.5), length.out = 6)[check$it]
cat("話者ごとに現れる条件の種類数:",
    unique(tapply(check$cond, check$sp, function(x) length(unique(x)))), "\n")

# 結果を「検定力 (モンテカルロSE, 異常フィット率)」の形で表示する補助関数。
fmt_power <- function(r) sprintf("%.2f (SE %.3f, 異常フィット率 %.0f%%)",
                                  r$power, r$se, r$anomaly_rate * 100)


■ 話者を増やすべきか、語を増やすべきか：

録音の手間は話者を増やすほうが圧倒的に大きい。同じ検定力を得られるなら、語を増やすほうが安上がりである。どちらが効くかを実際に計算して確かめる。

In [ ]:
# ⏳ 各設定で200回モデルを当てはめるので、このセルは数分かかる。
cat("【設計A】条件が「語の性質」である場合（voiced語 vs voiceless語）\n")
cat("  効果5ms、項目20語に固定して話者数を変える:\n")
for (n in c(10, 20, 40)) {
  cat(sprintf("    話者%2d名: 検定力 %s\n", n, fmt_power(simulate_power(n, 20, effect = 5))))
}
cat("  効果5ms、話者20名に固定して項目数を変える:\n")
for (k in c(10, 20, 40)) {
  cat(sprintf("    項目%2d語: 検定力 %s\n", k, fmt_power(simulate_power(20, k, effect = 5))))
}

# --- 実行結果 ---
# ⚠️ 査読で「収束警告・singular fitを一律に抑制し、そのp値をそのまま
#    成功・失敗へ数えている」との指摘を受け、simulate_power()を異常フィット率も
#    返す形に修正した（上のセル）。話者20名・項目20語・効果5msの1点は
#    再実行して確認済み（検定力0.57、モンテカルロSE 0.035、異常フィット率0%——
#    査読者が指摘した数値とも一致する）。それ以外の話者数・項目数の組み合わせは、
#    修正後の関数でまだ全点を再実行しきれていない。出版前に、この
#    ############部分を実際の出力（fmt_power()が返す検定力・SE・異常フィット率）
#    で置き換えること。
############################################################
#
# 【出力の読み方】
# ・検定力とは「効果が本当にあるとき、それを有意として検出できる確率」である。
#   慣習的な目標は0.80で、5回に4回は成功する設計を意味する。
# ・**表示にはモンテカルロ標準誤差（SE）と異常フィット率を必ず併記する。**
#   nsim=200程度のシミュレーションでは検定力0.5〜0.6付近でSEが0.03〜0.04にも
#   なるため、検定力を3桁目まで表示するのは過剰な精度である。
# ・**異常フィット率（収束警告・singular fitの起きた割合）も検定力と同時に見る。**
#   これが高ければ、その設定でのlmer()の当てはめ自体が不安定であることを
#   意味し、検定力の推定値も信用できない。0%に近ければ問題ないが、
#   高い場合は`sd_slope`を下げる、`nsim`を増やす、あるいはモデルの構造
#   そのものを見直す必要がある（後の演習3で、この率が跳ね上がる例を扱う）。
# ・話者数・項目数のどちらを増やすほうが効くかは、条件が語の性質そのもので
#   あるという設計（設計A）に固有の性質から決まる。次のセルで、
#   条件の入れ方を変える設計Bと比べる。


■ 設計そのものを変えると何が起きるか：

設計Aでは、voiced語とvoiceless語という**別々の語**を比べていた。これを、**同じ語を2つの条件で発話してもらう**設計に変えるとどうなるか。たとえば「同じ単語を、丁寧な発話とくだけた発話で読んでもらう」ような場合である。

In [ ]:
# 設計B: 条件を項目の内側で操作する（同じ語が両条件に現れる）
simulate_power_B <- function(n_speaker, n_item, effect,
                             sd_sp = 8, sd_slope = 6, sd_it = 4, sd_res = 7,
                             nsim = 200, seed = 2) {
  set.seed(seed)
  out <- replicate(nsim, {
    sp_int   <- rnorm(n_speaker, 0, sd_sp)
    sp_slope <- rnorm(n_speaker, 0, sd_slope)
    it_int   <- rnorm(n_item,    0, sd_it)
    # cond を設計に含めるので、同じ語が両条件に現れる
    d <- expand.grid(sp = 1:n_speaker, it = 1:n_item, cond = c(-0.5, 0.5))
    d$y <- 50 + effect * d$cond +
           sp_int[d$sp] + sp_slope[d$sp] * d$cond +
           it_int[d$it] + rnorm(nrow(d), 0, sd_res)
    d$sp <- factor(d$sp); d$it <- factor(d$it)
    # ★設計Aと同じく、異常フィットを黙って揉み消さない。
    anomaly <- FALSE
    m <- withCallingHandlers(
      tryCatch(lmer(y ~ cond + (1 + cond | sp) + (1 | it), data = d),
               error = function(e) NULL),
      warning = function(w) { anomaly <<- TRUE; invokeRestart("muffleWarning") }
    )
    if (is.null(m)) return(c(sig = NA, anomaly = 1))
    if (isSingular(m)) anomaly <- TRUE
    sig <- coef(summary(m))["cond", "Pr(>|t|)"] < 0.05
    c(sig = as.numeric(sig), anomaly = as.numeric(anomaly))
  })
  sig_vals <- out["sig", ]
  n_valid <- sum(!is.na(sig_vals))
  power <- mean(sig_vals, na.rm = TRUE)
  se <- sqrt(power * (1 - power) / n_valid)
  list(power = power, se = se, anomaly_rate = mean(out["anomaly", ]))
}

cat("【設計B】同じ語を両条件で発話してもらう場合\n")
cat("  効果5ms、項目20語に固定して話者数を変える:\n")
for (n in c(10, 20, 40)) {
  cat(sprintf("    話者%2d名: 検定力 %s\n", n, fmt_power(simulate_power_B(n, 20, effect = 5))))
}
cat("  効果5ms、話者20名に固定して項目数を変える:\n")
for (k in c(10, 20, 40)) {
  cat(sprintf("    項目%2d語: 検定力 %s\n", k, fmt_power(simulate_power_B(20, k, effect = 5))))
}

# --- 実行結果 ---
# 項目数を変えた側は実際に再実行して確認済み:
#   項目10語: 検定力 0.86 (SE 0.024, 異常フィット率 0%)
#   項目20語: 検定力 0.91 (SE 0.020, 異常フィット率 1%)
#   項目40語: 検定力 0.95 (SE 0.015, 異常フィット率 0%)
# ⚠️ 話者数を変えた側（10/20/40名）は、上記の異常フィット追跡版に修正した
#    あとまだ再実行していない。出版前に実行し、この############部分を
#    実際の出力で置き換えること。
############################################################
#
# 【出力の読み方】
# ・項目数を20語のまま揃えて確認できる範囲では、設計Bの検定力（0.91前後）は
#   設計Aの同条件（0.57）よりはっきり高い。理由は明快である。設計Bでは
#   同じ語が両条件に現れるので、**語ごとのばらつきが条件の比較から
#   差し引かれる**。語Aが本来長めであっても、条件1でも条件2でも同じだけ
#   長いので、その差を取れば消える。設計Aではこれができなかった。
# ・★実務上の結論は3つある。
#   (1) **可能なら、条件は項目の内側で操作する設計にする。**
#       同じ語を条件をまたいで使えるなら、それだけで検定力が大きく上がる。
#   (2) **設計が決まってから、増やすべき単位を決める。**
#       「話者を増やせばよい」「項目を増やせばよい」という一般則は無い。
#   (3) **どちらの設計でも、検定力はシミュレーションで確かめる。** 勘では
#       当てられない。異常フィット率も一緒に確認し、高ければ結果を疑う。
# ・★設計Aが悪いわけではない。voiced対voicelessのように、
#   **条件が語の性質そのものである場合は、設計Bは原理的に不可能**である。
#   そのときは、語数を十分に確保することが決定的に重要になる。


■ ★よくある誤り：実験のあとに検定力を計算する（事後検定力）：

「有意にならなかったので、検定力を計算したら0.3しかなかった。だからサンプル不足が原因である」——この推論は、**観測された効果量を使って計算する限り、何も情報を加えていない**。観測された効果量から計算した検定力はp値の単調な関数にすぎず、p値が大きければ必ず低く出るからである。

In [ ]:
# 観測された効果量を使った「事後検定力」が、p値と一対一に対応することを確かめる。
set.seed(88)
results <- map_dfr(1:60, function(i) {
  n <- 20
  x <- rep(c(-0.5, 0.5), length.out = n)
  y <- 0.4 * x + rnorm(n)          # 真の効果は 0.4 で固定
  tt <- t.test(y ~ factor(x))
  d_obs <- diff(rev(tapply(y, x, mean))) / sd(y)   # 観測された標準化効果量
  # 観測された効果量をそのまま使って計算した「事後検定力」
  pw <- power.t.test(n = n / 2, delta = abs(d_obs), sd = 1, sig.level = 0.05)$power
  tibble(p = tt$p.value, post_hoc_power = pw)
})

cat("p値と事後検定力の相関:", round(cor(results$p, results$post_hoc_power), 4), "\n")
cat("（順位相関）           :",
    round(cor(results$p, results$post_hoc_power, method = "spearman"), 4), "\n\n")
print(results %>% arrange(p) %>% slice(c(1, 2, 30, 59, 60)) %>%
        mutate(across(everything(), ~round(.x, 4))))

# --- 実行結果 ---
# p値と事後検定力の相関: -0.9021
# （順位相関）           : -0.9998
#
# # A tibble: 5 x 2
#        p post_hoc_power
#    <dbl>          <dbl>
# 1 0.0339         0.521
# 2 0.0358         0.497
# 3 0.312          0.164
# 4 0.922          0.0312
# 5 0.949          0.029
#
# 【出力の読み方】
# ・**順位相関が −0.9998 と、ほぼ完全に −1 である。** つまりp値が大きいほど
#   事後検定力は必ず小さくなり、両者の対応は事実上1対1に決まっている。
#   p = 0.0339 のとき検定力0.52、p = 0.949 のとき検定力0.029という具合である。
#   （ちょうど−1にならないのは、効果量の標準化に全体のSDを使っているためで、
#   検定が使う群内の統合SDとわずかにずれる。本質的な違いではない。）
# ・したがって「p値は有意でなかった。事後検定力を計算したら低かった。
#   だからサンプル不足だ」という主張は、**「p値が大きかったので、p値が
#   大きかった」と言っているのと同じ**である。新しい情報は1つも無い。
# ・★では有意にならなかったとき、どう報告すべきか。答えは
#   **信頼区間を示すこと**である。区間が狭くて0の近くにあるなら
#   「効果は小さい」と言え、区間が広いなら「まだ分からない」と言える。
#   この2つを区別できるのは信頼区間だけで、p値にも事後検定力にもできない。
# ・★検定力の計算が意味を持つのは、**実験を計画する段階で、
#   検出したい効果量を自分で決めて**行う場合だけである。
#   「先行研究の効果量を使えばよい」と考えたくなるが、これにも罠がある。
#   出版された研究は有意になったものに偏っているため、報告された効果量は
#   真の値より**大きめに出ている**（winner's curse）。
#   先行研究の効果量をそのまま使うと、必要な規模を過小に見積もることになる。
#   **「これ以上小さければ実質的に意味が無い」と自分で決めた値**を使うのが、
#   最も防御しやすい考え方である。


■ 演習：

1. 設計Aで、効果を5msから10msに変えたときの検定力を計算せよ。
2. 設計Aで検定力0.80を達成するには、話者20名のとき項目が何語必要か。
3. `sd_slope`（話者ごとの効果のばらつき）を6から2に下げると、検定力がどう変わるか確かめ、その理由を説明せよ。

■ 演習の解答：

In [ ]:
# 1. 効果が2倍になると検定力はどうなるか。
cat("設計A・話者20名・項目20語:\n")
cat("  効果 5ms:", fmt_power(simulate_power(20, 20, effect = 5)), "\n")
cat("  効果10ms:", fmt_power(simulate_power(20, 20, effect = 10)), "\n")

# 2. 項目数を増やして0.80を超える点を探す。
cat("\n設計A・話者20名・効果5ms で項目数を増やす:\n")
for (k in c(40, 60, 80, 120)) {
  cat(sprintf("  項目%3d語: 検定力 %s\n", k, fmt_power(simulate_power(20, k, effect = 5))))
}

# 3. 話者ごとの効果のばらつきを小さくする。
cat("\n設計A・話者20名・項目20語・効果5ms:\n")
cat("  sd_slope = 6:", fmt_power(simulate_power(20, 20, 5, sd_slope = 6)), "\n")
cat("  sd_slope = 2:", fmt_power(simulate_power(20, 20, 5, sd_slope = 2)), "\n")

# --- 実行結果 ---
# 設計A・話者20名・項目20語:
#   効果 5ms: 0.57 (SE 0.035, 異常フィット率 0%)
#   効果10ms: 0.98 (SE 0.010, 異常フィット率 0%)
#
# 設計A・話者20名・効果5ms で項目数を増やす:
#   項目 40語: 検定力 0.74 (SE 0.031, 異常フィット率 1%)
#   項目 60語: 検定力 0.78 (SE 0.030, 異常フィット率 0%)
#   項目 80語: 検定力 0.83 (SE 0.026, 異常フィット率 1%)
#   項目120語: 検定力 0.86 (SE 0.024, 異常フィット率 0%)
#
# 設計A・話者20名・項目20語・効果5ms:
#   sd_slope = 6: 0.57 (SE 0.035, 異常フィット率 0%)
#   sd_slope = 2: 0.73 (SE 0.031, 異常フィット率 22%)
#
# 【出力の読み方】
# ・1問目: 効果が5msから10msへ2倍になると、検定力は0.57から0.98へ跳ね上がる。
#   **検定力は効果量に対して急激に反応する。** だからこそ、
#   「どれくらいの効果を検出したいか」を決めることが、
#   サンプルサイズの計算そのものより重要になる。
#   ★ここを「先行研究では10msだったから10msで計算しよう」と済ませると、
#   実際の効果が5msだったときに設計が破綻する。**検出したい最小の効果**
#   （これより小さければ実質的に意味がない、と言える値）で計算するのが安全である。
# ・2問目: 項目を40語から120語へ3倍にしても、検定力は0.74から0.86までしか
#   上がらない。**増やすほど効きが鈍る（収穫逓減）。** 0.80を超えるのは
#   80語あたりである。0.86から0.95へ上げるコストは、0.60から0.80へ上げる
#   コストよりはるかに大きい。**どこで止めるかは、統計ではなく研究の
#   予算と価値判断で決める。**
# ・3問目: 話者ごとの効果のばらつき（sd_slope）を6から2へ下げると、
#   検定力は0.57から0.73へ上がる——サンプルサイズは1つも変えていない。
#   **ただし、この設定（sd_slope = 2）では異常フィット率が22%まで跳ね上がる。**
#   200回中およそ44回で収束警告かsingular fitが起きている。sd_slopeを
#   小さくしすぎると、データの中に話者ごとの傾きのばらつきがほとんど無くなり、
#   モデルが「ランダム傾きの分散をほぼ0と推定する」ケースが増えるためである。
#   **この22%を無視して検定力0.73だけを報告するのは危険である。** 5回に
#   1回以上のペースで当てはめ自体が不安定な状況を含む数字だからである。
#   異常フィット率が高い設定では、ランダム効果構造を簡略化する
#   （第5回参照）か、`nsim`を増やして異常フィットの影響を薄めるかを検討する。
# ・なぜsd_slopeを下げると検定力が上がるのか。検定したいのは「平均的な効果が
#   0か否か」だが、**話者ごとに効果の大きさが違えば、その平均の推定も
#   不安定になる**。全員がほぼ同じ大きさの効果を示すなら、少ない話者数でも
#   平均を精度よく決められる——ただし上で見たとおり、その代償として
#   モデルの当てはめ自体が不安定になりやすい場合がある。
# ・★実務的な含意が2つある。第一に、**均質な話者集団を選ぶと検定力が
#   上がりうる**。方言・年齢・性別を揃えれば効果のばらつきは減る。ただし
#   一般化できる範囲が狭まるだけでなく、モデルの当てはめそのものが
#   不安定になりうることも今回わかった。第二に、**録音条件を統制することも
#   検定力を上げる**。雑音や機材の違いは残差（sd_res）を膨らませ、
#   同じように検定力を削る。**サンプルサイズだけが検定力を決めるのではない。**
